In [ ]:
# This class never prices options directly — it only provides vols.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Methods_xlWings import *

In [ ]:
class ImpliedVolSurface:
    def __init__(self):
        pass

    def vol(self, strike, maturity):
        raise NotImplementedError("Must implement vol(K, T)")

    def total_variance(self, strike, maturity):
        sigma = self.vol(strike, maturity)
        return sigma**2 * maturity
    

In [ ]:
class SVI(ImpliedVolSurface):
    def __init__(self, a, b, rho, m, sigma, forward):
        self.a = a
        self.b = b
        self.rho = rho
        self.m = m
        self.sigma = sigma
        self.forward = forward

    def total_variance(self, strike, maturity):
        k = np.log(strike / self.forward)
        return (
            self.a
            + self.b * (
                self.rho * (k - self.m)
                + np.sqrt((k - self.m)**2 + self.sigma**2)
            )
        )

    def vol(self, strike, maturity):
        w = self.total_variance(strike, maturity)
        return np.sqrt(np.maximum(w / maturity, 1e-12))
        

In [ ]:
class SABR(ImpliedVolSurface):
    def __init__(self, alpha, beta, rho, nu, forward):
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.nu = nu
        self.forward = forward

    def vol(self, strike, maturity):
        F = self.forward
        K = strike

        if np.isclose(F, K):
            # ATM formula
            term1 = self.alpha / (F**(1 - self.beta))
            term2 = (
                ((1 - self.beta)**2 / 24) * (self.alpha**2 / (F**(2 - 2*self.beta)))
                + (self.rho * self.beta * self.nu * self.alpha) / (4 * F**(1 - self.beta))
                + ((2 - 3*self.rho**2) * self.nu**2 / 24)
            ) * maturity
            return term1 * (1 + term2)

       logFK = np.log(F / K)
        z = (self.nu / self.alpha) * (F*K)**((1 - self.beta)/2) * logFK
        xz = np.log((np.sqrt(1 - 2*self.rho*z + z**2) + z - self.rho) / (1 - self.rho))

        A = self.alpha / ((F*K)**((1 - self.beta)/2))
        B = z / xz

        correction = (
            1
            + (
                ((1 - self.beta)**2 / 24) * (self.alpha**2 / (F*K)**(1 - self.beta))
                + (self.rho * self.beta * self.nu * self.alpha) / (4 * (F*K)**((1 - self.beta)/2))
                + ((2 - 3*self.rho**2) * self.nu**2 / 24)
            ) * maturity
        )

        return A * B * correction
